In [ ]:
# Class vs instance attributes
# Instance attributes live on each object. Class attributes are shared. The classic bug: mutable class attributes shared by accident.

class BankAccount:
    interest_rate = 0.05 # class attr - shared by all instances

    def __init__(self, owner:str, balance: float = 0):
        self.owner = owner # instance attr - unique to each instance
        self._balance = balance # instance attr - unique to each instance
    
    @classmethod
    def from_dict(cls, data: dict):
        return cls(owner=data['owner'], balance=data.get('balance', 0))
    
    @staticmethod
    def validate_amount(amount: float) -> bool:
        return amount > 0 

In [ ]:
#The mutable default gotcha
#Mutable class attributes (lists, dicts) are shared — one of the most common intermediate-level bugs.

class BadCard:
    items = [] # # DANGER: shared across all instances

    def add(self,item): 
        self.items.append(item)

class GoodCart:
    def __init__(self):
        self.items = [] # instance attr - unique to each instance

    def add(self, item):
        self.items.append(item)

In [3]:
c1 , c2 = BadCard(), BadCard()
c1.add('apple')
print(c1.items) # ['apple']
print(c2.items) # ['apple'] - unexpected! shared across instances

['apple']
['apple']


Great question — this trips up a lot of intermediate devs. Here's the core difference:

| | `@classmethod` | `@staticmethod` |
|---|---|---|
| First param | `cls` (the class itself) | nothing special |
| Knows about the class? | Yes | No |
| Knows about instances? | No | No |
| Can be inherited? | Yes, `cls` adapts | Yes, but no class awareness |
| Main use | Alternative constructors, factory methods | Utility/helper functions |

**Simple rule of thumb:**
- Need `cls` to create or inspect the class → `@classmethod`
- Just a helper function that happens to belong here → `@staticmethod`
- Need `self` to access instance data → regular method

In [ ]:
#@staticmethod — just a regular function that lives inside a class for organisation. It doesn't know what class it's on.
class MathUtils:
    @staticmethod
    def add(a, b):
        return a + b   # no self, no cls — pure function

MathUtils.add(3, 4)    # 7

7

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name 

    @classmethod
    def create(cls, name):      # cls = whatever class called this
        return cls(name)        # creates an instance of THAT class   

class Dog(Animal):
    def speak(self):
        return f"Woof! I'm {self.name}"

d = Dog.create("Rex")           # cls is Dog, so returns a Dog
print(type(d))                  # <class 'Dog'>
print(d.speak())                # Woof! I'm Rex

#If you used @staticmethod here and hardcoded Animal(name), Dog.create() would return an Animal, not a Dog — breaking inheritance.


<class '__main__.Dog'>
Woof! I'm Rex


In [6]:
# The most common real-world use: alternative constructors
from datetime import date

class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    @classmethod
    def from_birth_year(cls, name, birth_year):   # construct from different data
        age = date.today().year - birth_year
        return cls(name, age)

    @classmethod
    def from_dict(cls, data: dict):               # construct from a dict
        return cls(data["name"], data["age"])

    @staticmethod
    def is_adult(age: int) -> bool:               # utility — doesn't need class
        return age >= 18

p1 = Person("Priya", 28)
p2 = Person.from_birth_year("Raj", 1995)
p3 = Person.from_dict({"name": "Sam", "age": 22})

print(Person.is_adult(16))   # False

False


## INheritance, Polymorphism

In [ ]:
#Inheritance & `super()`
# Use inheritance for is-a relationships. `super()` lets you extend, not replace, parent behaviour. 
# Python resolves method lookup via MRO (Method Resolution Order).
